<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/OMI_Performance_Ceiling_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import h5py

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

ROOT = Path("/content/drive/MyDrive/ECG_Project")

VAL_H5 = (
    ROOT /
    "results/preprocessing/processed/validation.h5"
)

VAL_CSV = (
    ROOT /
    "results/preprocessing/processed/validation_omi.csv"
)

# CHANGE THIS
CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/ECG_Project/OMI_model/ecgfounder_omi_last2.pth"
)

OUT = (
    ROOT /
    "results/ceiling_analysis"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTION_FILE = (
    OUT /
    "plain_ecgfounder_validation_predictions.csv"
)

print("Validation H5 :", VAL_H5.exists())
print("Validation CSV:", VAL_CSV.exists())
print("Checkpoint    :", CHECKPOINT_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
Validation H5 : True
Validation CSV: True
Checkpoint    : True


In [ ]:
import torch
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import numpy as np
from collections import Counter
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data, label):
        self.data = data
        self.label = label

    def __getitem__(self, index):
        return (torch.tensor(self.data[index], dtype=torch.float), torch.tensor(self.label[index], dtype=torch.long))

    def __len__(self):
        return len(self.data)

class MyConv1dPadSame(nn.Module):
    """
    extend nn.Conv1d to support SAME padding

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super(MyConv1dPadSame, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups = groups
        self.conv = torch.nn.Conv1d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

    def forward(self, x):

        net = x

        # compute pad shape
        in_dim = net.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)
        # print(net.shape)
        net = self.conv(net)

        return net

class MyMaxPool1dPadSame(nn.Module):
    """
    extend nn.MaxPool1d to support SAME padding

    params:
        kernel_size: kernel size
        stride: the stride of the window. Default value is kernel_size

    input: (n_sample, n_channel, n_length)
    """
    def __init__(self, kernel_size):
        super(MyMaxPool1dPadSame, self).__init__()
        self.kernel_size = kernel_size
        self.max_pool = torch.nn.MaxPool1d(kernel_size=self.kernel_size)

    def forward(self, x):

        net = x

        # compute pad shape
        p = max(0, self.kernel_size - 1)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)

        net = self.max_pool(net)

        return net

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class BasicBlock(nn.Module):
    """
    Basic Block:
        conv1 -> convk -> conv1

    params:
        in_channels: number of input channels
        out_channels: number of output channels
        ratio: ratio of channels to out_channels
        kernel_size: kernel window length
        stride: kernel step size
        groups: number of groups in convk
        downsample: whether downsample length
        use_bn: whether use batch_norm
        use_do: whether use dropout

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, downsample, is_first_block=False, use_bn=True, use_do=True):
        super(BasicBlock, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.downsample = downsample
        self.stride = stride if self.downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        self.middle_channels = int(self.out_channels * self.ratio)

        # the first conv, conv1
        self.bn1 = nn.BatchNorm1d(in_channels)
        self.activation1 = Swish()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(
            in_channels=self.in_channels,
            out_channels=self.middle_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # the second conv, convk
        self.bn2 = nn.BatchNorm1d(self.middle_channels)
        self.activation2 = Swish()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.middle_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

        # the third conv, conv1
        self.bn3 = nn.BatchNorm1d(self.middle_channels)
        self.activation3 = Swish()
        self.do3 = nn.Dropout(p=0.5)
        self.conv3 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.out_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # Squeeze-and-Excitation
        r = 2
        self.se_fc1 = nn.Linear(self.out_channels, self.out_channels//r)
        self.se_fc2 = nn.Linear(self.out_channels//r, self.out_channels)
        self.se_activation = Swish()

        if self.downsample:
            self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):

        identity = x

        out = x
        # the first conv, conv1
        if not self.is_first_block:
            if self.use_bn:
                out = self.bn1(out)
            out = self.activation1(out)
            if self.use_do:
                out = self.do1(out)
        out = self.conv1(out)

        # the second conv, convk
        if self.use_bn:
            out = self.bn2(out)
        out = self.activation2(out)
        if self.use_do:
            out = self.do2(out)
        out = self.conv2(out)

        # the third conv, conv1
        if self.use_bn:
            out = self.bn3(out)
        out = self.activation3(out)
        if self.use_do:
            out = self.do3(out)
        out = self.conv3(out) # (n_sample, n_channel, n_length)

        # Squeeze-and-Excitation
        se = out.mean(-1) # (n_sample, n_channel)
        se = self.se_fc1(se)
        se = self.se_activation(se)
        se = self.se_fc2(se)
        se = torch.sigmoid(se) # (n_sample, n_channel)
        out = torch.einsum('abc,ab->abc', out, se)

        # if downsample, also downsample identity
        if self.downsample:
            identity = self.max_pool(identity)

        # if expand channel, also pad zeros to identity
        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1,-2)
            ch1 = (self.out_channels-self.in_channels)//2
            ch2 = self.out_channels-self.in_channels-ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1,-2)

        # shortcut
        out += identity

        return out

class BasicStage(nn.Module):
    """
    Basic Stage:
        block_1 -> block_2 -> ... -> block_M
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, i_stage, m_blocks, use_bn=True, use_do=True, verbose=False):
        super(BasicStage, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.i_stage = i_stage
        self.m_blocks = m_blocks
        self.use_bn = use_bn
        self.use_do = use_do
        self.verbose = verbose

        self.block_list = nn.ModuleList()
        for i_block in range(self.m_blocks):

            # first block
            if self.i_stage == 0 and i_block == 0:
                self.is_first_block = True
            else:
                self.is_first_block = False
            # downsample, stride, input
            if i_block == 0:
                self.downsample = True
                self.stride = stride
                self.tmp_in_channels = self.in_channels
            else:
                self.downsample = False
                self.stride = 1
                self.tmp_in_channels = self.out_channels

            # build block
            tmp_block = BasicBlock(
                in_channels=self.tmp_in_channels,
                out_channels=self.out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=self.groups,
                downsample=self.downsample,
                is_first_block=self.is_first_block,
                use_bn=self.use_bn,
                use_do=self.use_do)
            self.block_list.append(tmp_block)

    def forward(self, x):

        out = x

        for i_block in range(self.m_blocks):
            net = self.block_list[i_block]
            out = net(out)
            if self.verbose:
                print('stage: {}, block: {}, in_channels: {}, out_channels: {}, outshape: {}'.format(self.i_stage, i_block, net.in_channels, net.out_channels, list(out.shape)))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv1.in_channels, net.conv1.out_channels, net.conv1.kernel_size, net.conv1.stride, net.conv1.groups))
                print('stage: {}, block: {}, convk: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv2.in_channels, net.conv2.out_channels, net.conv2.kernel_size, net.conv2.stride, net.conv2.groups))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv3.in_channels, net.conv3.out_channels, net.conv3.kernel_size, net.conv3.stride, net.conv3.groups))

        return out

class Net1D(nn.Module):
    """

    Input:
        X: (n_samples, n_channel, n_length)
        Y: (n_samples)

    Output:
        out: (n_samples)

    params:
        in_channels
        base_filters
        filter_list: list, filters for each stage
        m_blocks_list: list, number of blocks of each stage
        kernel_size
        stride
        groups_width
        n_stages
        n_classes
        use_bn
        use_do

    """

    def __init__(self, in_channels, base_filters, ratio, filter_list, m_blocks_list, kernel_size, stride, groups_width, n_classes, use_bn=True, use_do=True, return_features=False, verbose=False):
        super(Net1D, self).__init__()

        self.in_channels = in_channels
        self.base_filters = base_filters
        self.ratio = ratio
        self.filter_list = filter_list
        self.m_blocks_list = m_blocks_list
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups_width = groups_width
        self.n_stages = len(filter_list)
        self.n_classes = n_classes
        self.use_bn = use_bn
        self.use_do = use_do
        self.return_features = return_features
        self.verbose = verbose

        # first conv
        self.first_conv = MyConv1dPadSame(
            in_channels=in_channels,
            out_channels=self.base_filters,
            kernel_size=self.kernel_size,
            stride=2)
        self.first_bn = nn.BatchNorm1d(base_filters)
        self.first_activation = Swish()

        # stages
        self.stage_list = nn.ModuleList()
        in_channels = self.base_filters
        for i_stage in range(self.n_stages):

            out_channels = self.filter_list[i_stage]
            m_blocks = self.m_blocks_list[i_stage]
            tmp_stage = BasicStage(
                in_channels=in_channels,
                out_channels=out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=out_channels//self.groups_width,
                i_stage=i_stage,
                m_blocks=m_blocks,
                use_bn=self.use_bn,
                use_do=self.use_do,
                verbose=self.verbose)
            self.stage_list.append(tmp_stage)
            in_channels = out_channels

        # final prediction
        self.dense = nn.Linear(in_channels, n_classes)

    def forward(self, x):

        out = x

        # first conv
        out = self.first_conv(out)
        if self.use_bn:
            out = self.first_bn(out)
        out = self.first_activation(out)

        # stages
        for i_stage in range(self.n_stages):
            net = self.stage_list[i_stage]
            out = net(out)

        # final prediction
        deep_features = out.mean(-1)
        out = self.dense(deep_features)

        if self.return_features:
            return out, deep_features
        else:
            return out

if __name__ == "__main__":
  model = Net1D(
      in_channels=12,
      base_filters=64, #32 64
      ratio=1,
      filter_list=[64,160,160,400,400,1024,1024],    #[16,32,32,80,80,256,256] [32,64,64,160,160,512,512] [64,160,160,400,400,1024,1024]
      m_blocks_list=[2,2,2,3,3,4,4],   #[2,2,2,2,2,2,2] [2,2,2,3,3,4,4]
      kernel_size=16,
      stride=2,
      groups_width=16,
      verbose=True,
      use_bn=False,
      use_do=False,
      n_classes=1)
  print(model)



Net1D(
  (first_conv): MyConv1dPadSame(
    (conv): Conv1d(12, 64, kernel_size=(16,), stride=(2,))
  )
  (first_bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (first_activation): Swish()
  (stage_list): ModuleList(
    (0): BasicStage(
      (block_list): ModuleList(
        (0): BasicBlock(
          (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation1): Swish()
          (do1): Dropout(p=0.5, inplace=False)
          (conv1): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
          )
          (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation2): Swish()
          (do2): Dropout(p=0.5, inplace=False)
          (conv2): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(16,), stride=(2,), groups=4)
          )
          (bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [ ]:
# Build exact ECGFounder architecture
model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=1,
    use_bn=False,
    use_do=False,
    return_features=True,
    verbose=False
)


In [ ]:
# ============================================================
# CELL 6 — H5 DATASET FOR OMI
# ============================================================

import h5py
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

BASE = "/content/drive/MyDrive/ECG_Project/results/preprocessing/processed"

TRAIN_H5  = f"{BASE}/train_fixed.h5"
VAL_H5    = f"{BASE}/validation.h5"

TRAIN_CSV = f"{BASE}/train_omi.csv"
VAL_CSV   = f"{BASE}/validation_omi.csv"


class OMIDataset(Dataset):

    def __init__(self, h5_path, csv_path):

        self.h5_path = h5_path
        self.df = pd.read_csv(csv_path)

        # Read IDs once for verification
        with h5py.File(self.h5_path, "r") as f:

            ids = f["record_ids"][:]

            self.record_ids = [
                x.decode() if isinstance(x, bytes) else str(x)
                for x in ids
            ]

            self.length = len(f["signals"])

        # Safety checks
        assert self.length == len(self.df), (
            f"H5={self.length}, CSV={len(self.df)}"
        )

        csv_ids = self.df["record_id"].astype(str).str.zfill(5).tolist()
        h5_ids  = [str(x).zfill(5) for x in self.record_ids]

        assert csv_ids == h5_ids, "CSV and H5 record order mismatch!"

        self.labels = self.df["OMI"].astype(np.float32).values

        self.h5 = None

        print(
            f"Loaded {self.length} ECGs | "
            f"OMI={int(self.labels.sum())} | "
            f"non-OMI={int(len(self.labels)-self.labels.sum())}"
        )


    def __len__(self):
        return self.length


    def __getitem__(self, idx):

        # Open H5 lazily
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, "r")

        ecg = self.h5["signals"][idx]

        ecg = torch.tensor(
            ecg,
            dtype=torch.float32
        )

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        return ecg, label

In [ ]:
# ============================================================
# CELL 7 — DATALOADERS
# ============================================================

train_dataset = OMIDataset(
    TRAIN_H5,
    TRAIN_CSV
)

val_dataset = OMIDataset(
    VAL_H5,
    VAL_CSV
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Loaded 15273 ECGs | OMI=985 | non-OMI=14288
Loaded 2685 ECGs | OMI=166 | non-OMI=2519

Train batches: 955
Validation batches: 168


In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

clean_state = {}

for k, v in state_dict.items():

    if k.startswith("module."):
        k = k[7:]

    clean_state[k] = v

model.load_state_dict(
    clean_state,
    strict=True
)

model = model.to(DEVICE)
model.eval()

print("Checkpoint loaded successfully.")
print("Saved epoch:", checkpoint["epoch"])
print("Saved AUROC:", checkpoint["val_auroc"])
print("Saved AUPRC:", checkpoint["val_auprc"])

Checkpoint loaded successfully.
Saved epoch: 6
Saved AUROC: 0.8788891174065057
Saved AUPRC: 0.4000248638802949


In [ ]:
all_probs = []
all_labels = []

model.eval()

with torch.no_grad():

    for batch in val_loader:

        # Most likely format
        if len(batch) == 2:
            x, y = batch

        else:
            x = batch[0]
            y = batch[1]

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.float().view(-1)

        output = model(x)

        if isinstance(output, tuple):
            logits = output[0]
        else:
            logits = output

        logits = logits.view(-1)

        probs = torch.sigmoid(
            logits
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            y.cpu().numpy()
        )


all_probs = np.asarray(all_probs)

all_labels = (
    np.asarray(all_labels)
    .astype(int)
)

print("Predictions:", len(all_probs))
print("Labels:", len(all_labels))
print("Positive labels:", all_labels.sum())

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predictions: 2685
Labels: 2685
Positive labels: 166


In [ ]:
val_meta = pd.read_csv(
    VAL_CSV,
    dtype={
        "record_id": str,
        "Patient_id": str
    }
)

print("Metadata:", len(val_meta))

assert len(val_meta) == len(all_probs)

metadata_labels = (
    val_meta["OMI"]
    .astype(int)
    .values
)

agreement = np.mean(
    metadata_labels ==
    all_labels
)

print(
    "Label/order agreement:",
    f"{agreement*100:.2f}%"
)

assert agreement == 1.0, (
    "Validation order mismatch!"
)


pred_df = pd.DataFrame({

    "record_id":
        val_meta["record_id"],

    "Patient_id":
        val_meta["Patient_id"],

    "true_OMI":
        metadata_labels,

    "omi_prob":
        all_probs
})


pred_df.to_csv(
    PREDICTION_FILE,
    index=False
)

print("\nSaved:")
print(PREDICTION_FILE)

print(pred_df.head())

Metadata: 2685
Label/order agreement: 100.00%

Saved:
/content/drive/MyDrive/ECG_Project/results/ceiling_analysis/plain_ecgfounder_validation_predictions.csv
  record_id Patient_id  true_OMI  omi_prob
0     12776     P00004         0  0.005991
1     03019     P00009         0  0.025167
2     05748     P00019         0  0.383201
3     03326     P00024         0  0.026385
4     02721     P00034         0  0.011690


In [ ]:
print("Metadata rows :", len(val_meta))
print("Predictions   :", len(all_probs))
print("Labels        :", len(all_labels))
print("Val dataset   :", len(val_dataset))
print("Val loader batches:", len(val_loader))

Metadata rows : 2685
Predictions   : 2685
Labels        : 2685
Val dataset   : 2685
Val loader batches: 168


ACTUAL ANALYSIS

In [ ]:
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/ECG_Project"

MASTER_PATH = (
    f"{ROOT}/results/ceiling_analysis/"
    "master_analysis_table.csv"
)

PRED_PATH = (
    f"{ROOT}/results/ceiling_analysis/"
    "plain_ecgfounder_validation_predictions.csv"
)

# ---------------------------------------------------------
# Load files
# ---------------------------------------------------------

master = pd.read_csv(
    MASTER_PATH,
    dtype={
        "record_id": str,
        "Patient_id": str
    }
)

pred = pd.read_csv(
    PRED_PATH,
    dtype={
        "record_id": str,
        "Patient_id": str
    }
)

# Standardize record IDs
master["record_id"] = master["record_id"].astype(str).str.zfill(5)
pred["record_id"] = pred["record_id"].astype(str).str.zfill(5)

# ---------------------------------------------------------
# Merge
# ---------------------------------------------------------

df = pred.merge(
    master,
    on="record_id",
    how="left",
    suffixes=("_pred", "")
)

print("Merged ECGs:", len(df))
print("True OMI:", df["true_OMI"].sum())


# ---------------------------------------------------------
# Keep only true OMI patients
# ---------------------------------------------------------

omi = df[df["true_OMI"] == 1].copy()

print("\nTotal OMI cases:", len(omi))


# ---------------------------------------------------------
# STEMI-OMI
# ---------------------------------------------------------

stemi_omi = omi[
    (omi["STEMI"] == 1) &
    (omi["NSTEMI"] == 0)
]

# NSTEMI-OMI
nstemi_omi = omi[
    (omi["NSTEMI"] == 1) &
    (omi["STEMI"] == 0)
]


def show_scores(name, data):

    print(f"\n{name}")
    print("-" * 40)

    print("Cases :", len(data))

    print(
        "Mean probability   :",
        round(data["omi_prob"].mean(), 4)
    )

    print(
        "Median probability :",
        round(data["omi_prob"].median(), 4)
    )

    print(
        "25th percentile    :",
        round(data["omi_prob"].quantile(0.25), 4)
    )

    print(
        "75th percentile    :",
        round(data["omi_prob"].quantile(0.75), 4)
    )


show_scores(
    "STEMI-OMI",
    stemi_omi
)

show_scores(
    "NSTEMI-OMI",
    nstemi_omi
)

Merged ECGs: 2685
True OMI: 166

Total OMI cases: 166

STEMI-OMI
----------------------------------------
Cases : 99
Mean probability   : 0.7364
Median probability : 0.8694
25th percentile    : 0.5538
75th percentile    : 0.9654

NSTEMI-OMI
----------------------------------------
Cases : 67
Mean probability   : 0.4899
Median probability : 0.5056
25th percentile    : 0.2291
75th percentile    : 0.7443


In [ ]:
def vessel_stats(name, column):

    group = omi[omi[column] == 1]

    print(f"\n{name}")
    print("-" * 35)

    print("Cases :", len(group))

    print(
        "Mean probability   :",
        round(group["omi_prob"].mean(), 4)
    )

    print(
        "Median probability :",
        round(group["omi_prob"].median(), 4)
    )

    print(
        "25th percentile    :",
        round(group["omi_prob"].quantile(0.25), 4)
    )

    print(
        "75th percentile    :",
        round(group["omi_prob"].quantile(0.75), 4)
    )


vessel_stats("LAD OMI", "LAD_GROUP")
vessel_stats("LCX OMI", "LCX_GROUP")
vessel_stats("RCA OMI", "RCA_GROUP")


LAD OMI
-----------------------------------
Cases : 66
Mean probability   : 0.6863
Median probability : 0.8203
25th percentile    : 0.4256
75th percentile    : 0.9679

LCX OMI
-----------------------------------
Cases : 53
Mean probability   : 0.6104
Median probability : 0.6876
25th percentile    : 0.3733
75th percentile    : 0.9295

RCA OMI
-----------------------------------
Cases : 83
Mean probability   : 0.6461
Median probability : 0.7476
25th percentile    : 0.384
75th percentile    : 0.9371


In [ ]:
omi = omi.copy()

omi["n_vessels"] = (
    omi["LM_GROUP"] +
    omi["LAD_GROUP"] +
    omi["LCX_GROUP"] +
    omi["RCA_GROUP"]
)

single = omi[omi["n_vessels"] == 1]
multi  = omi[omi["n_vessels"] >= 2]

def show_group(name, data):

    print(f"\n{name}")
    print("-" * 35)

    print("Cases :", len(data))

    print(
        "Mean probability   :",
        round(data["omi_prob"].mean(), 4)
    )

    print(
        "Median probability :",
        round(data["omi_prob"].median(), 4)
    )

    print(
        "25th percentile    :",
        round(data["omi_prob"].quantile(0.25), 4)
    )

    print(
        "75th percentile    :",
        round(data["omi_prob"].quantile(0.75), 4)
    )


show_group(
    "Single-vessel OMI",
    single
)

show_group(
    "Multi-vessel OMI",
    multi
)


Single-vessel OMI
-----------------------------------
Cases : 134
Mean probability   : 0.6218
Median probability : 0.6768
25th percentile    : 0.3664
75th percentile    : 0.9308

Multi-vessel OMI
-----------------------------------
Cases : 32
Mean probability   : 0.6999
Median probability : 0.8534
25th percentile    : 0.4725
75th percentile    : 0.9485


In [ ]:
groups = []

for presentation, condition in [
    ("STEMI", (omi["STEMI"] == 1)),
    ("NSTEMI", (omi["NSTEMI"] == 1))
]:

    for vessel, column in [
        ("LAD", "LAD_GROUP"),
        ("LCX", "LCX_GROUP"),
        ("RCA", "RCA_GROUP")
    ]:

        g = omi[
            condition &
            (omi[column] == 1)
        ]

        groups.append({
            "Presentation": presentation,
            "Vessel": vessel,
            "Cases": len(g),
            "Mean_prob": g["omi_prob"].mean(),
            "Median_prob": g["omi_prob"].median()
        })


result = pd.DataFrame(groups)

print(
    result.round(4).to_string(index=False)
)

Presentation Vessel  Cases  Mean_prob  Median_prob
       STEMI    LAD     47     0.7047       0.8422
       STEMI    LCX     26     0.7681       0.8959
       STEMI    RCA     47     0.7722       0.8827
      NSTEMI    LAD     19     0.6408       0.7581
      NSTEMI    LCX     27     0.4586       0.4823
      NSTEMI    RCA     36     0.4815       0.4497


In [ ]:
print("Saved epoch :", checkpoint.get("epoch"))
print("Val AUROC   :", checkpoint.get("val_auroc"))
print("Val AUPRC   :", checkpoint.get("val_auprc"))

print("\nCONFIG:")
print(checkpoint.get("config"))

Saved epoch : 6
Val AUROC   : 0.8788891174065057
Val AUPRC   : 0.4000248638802949

CONFIG:
{'unfreeze_mode': 'last2', 'head_lr': 0.0001, 'backbone_lr': 1e-05, 'weight_decay': 0.0001, 'epochs': 20, 'max_grad_norm': 5.0, 'save_dir': '/content/drive/MyDrive/ECG_Project/OMI_model'}


In [ ]:
config = checkpoint.get("config", {})

for k, v in config.items():
    if "threshold" in str(k).lower() or "thresh" in str(k).lower():
        print(k, ":", v)

In [ ]:
from sklearn.metrics import confusion_matrix

THRESHOLD = 0.5

y_true = pred["true_OMI"].values
y_prob = pred["omi_prob"].values

y_pred = (y_prob >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

precision = tp / (tp + fp)
f1 = (
    2 * precision * sensitivity /
    (precision + sensitivity)
)

print("Threshold   :", THRESHOLD)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print("\nSensitivity :", round(sensitivity, 4))
print("Specificity :", round(specificity, 4))
print("Precision   :", round(precision, 4))
print("F1          :", round(f1, 4))

Threshold   : 0.5

TN: 2205
FP: 314
FN: 55
TP: 111

Sensitivity : 0.6687
Specificity : 0.8753
Precision   : 0.2612
F1          : 0.3756


In [ ]:
THRESHOLD = 0.5

omi = omi.copy()

omi["predicted_OMI"] = (
    omi["omi_prob"] >= THRESHOLD
).astype(int)


def detection_stats(name, data):

    total = len(data)

    detected = int(
        data["predicted_OMI"].sum()
    )

    missed = total - detected

    sensitivity = (
        detected / total
        if total > 0
        else np.nan
    )

    print(f"\n{name}")
    print("-" * 40)

    print("Total     :", total)
    print("Detected  :", detected)
    print("Missed    :", missed)

    print(
        "Sensitivity:",
        round(sensitivity, 4)
    )


# Overall presentation
detection_stats(
    "STEMI-OMI",
    omi[omi["STEMI"] == 1]
)

detection_stats(
    "NSTEMI-OMI",
    omi[omi["NSTEMI"] == 1]
)


# NSTEMI by vessel
nstemi = omi[
    omi["NSTEMI"] == 1
]

detection_stats(
    "NSTEMI-LAD OMI",
    nstemi[nstemi["LAD_GROUP"] == 1]
)

detection_stats(
    "NSTEMI-LCX OMI",
    nstemi[nstemi["LCX_GROUP"] == 1]
)

detection_stats(
    "NSTEMI-RCA OMI",
    nstemi[nstemi["RCA_GROUP"] == 1]
)


STEMI-OMI
----------------------------------------
Total     : 99
Detected  : 77
Missed    : 22
Sensitivity: 0.7778

NSTEMI-OMI
----------------------------------------
Total     : 67
Detected  : 34
Missed    : 33
Sensitivity: 0.5075

NSTEMI-LAD OMI
----------------------------------------
Total     : 19
Detected  : 13
Missed    : 6
Sensitivity: 0.6842

NSTEMI-LCX OMI
----------------------------------------
Total     : 27
Detected  : 13
Missed    : 14
Sensitivity: 0.4815

NSTEMI-RCA OMI
----------------------------------------
Total     : 36
Detected  : 17
Missed    : 19
Sensitivity: 0.4722


In [ ]:
THRESHOLD = 0.5

df["predicted_OMI"] = (
    df["omi_prob"] >= THRESHOLD
).astype(int)

# True non-OMI only
non_omi = df[df["true_OMI"] == 0].copy()

false_positive = non_omi[
    non_omi["predicted_OMI"] == 1
]

true_negative = non_omi[
    non_omi["predicted_OMI"] == 0
]

print("False positives:", len(false_positive))
print("True negatives :", len(true_negative))


def compare_feature(name, column):

    fp_n = int(false_positive[column].sum())
    tn_n = int(true_negative[column].sum())

    fp_pct = 100 * fp_n / len(false_positive)
    tn_pct = 100 * tn_n / len(true_negative)

    print(f"\n{name}")
    print("-" * 35)

    print(
        f"False positives: {fp_n} "
        f"({fp_pct:.2f}%)"
    )

    print(
        f"True negatives : {tn_n} "
        f"({tn_pct:.2f}%)"
    )


compare_feature("STEMI", "STEMI")
compare_feature("NSTEMI", "NSTEMI")

compare_feature("LAD", "LAD_GROUP")
compare_feature("LCX", "LCX_GROUP")
compare_feature("RCA", "RCA_GROUP")

False positives: 314
True negatives : 2205

STEMI
-----------------------------------
False positives: 80 (25.48%)
True negatives : 39 (1.77%)

NSTEMI
-----------------------------------
False positives: 29 (9.24%)
True negatives : 70 (3.17%)

LAD
-----------------------------------
False positives: 31 (9.87%)
True negatives : 40 (1.81%)

LCX
-----------------------------------
False positives: 17 (5.41%)
True negatives : 31 (1.41%)

RCA
-----------------------------------
False positives: 24 (7.64%)
True negatives : 38 (1.72%)


In [ ]:
# ============================================================
# HIGH-CONFIDENCE ERRORS
# ============================================================

high_conf_fn = df[
    (df["true_OMI"] == 1) &
    (df["omi_prob"] <= 0.20)
].copy()

high_conf_fp = df[
    (df["true_OMI"] == 0) &
    (df["omi_prob"] >= 0.80)
].copy()


print("High-confidence false negatives:", len(high_conf_fn))
print("High-confidence false positives:", len(high_conf_fp))


# ------------------------------------------------------------
# HIGH-CONFIDENCE FALSE NEGATIVES
# ------------------------------------------------------------

print("\n========== HIGH-CONFIDENCE FN ==========")

print(
    "STEMI:",
    int(high_conf_fn["STEMI"].sum())
)

print(
    "NSTEMI:",
    int(high_conf_fn["NSTEMI"].sum())
)

print(
    "LAD:",
    int(high_conf_fn["LAD_GROUP"].sum())
)

print(
    "LCX:",
    int(high_conf_fn["LCX_GROUP"].sum())
)

print(
    "RCA:",
    int(high_conf_fn["RCA_GROUP"].sum())
)

print(
    "\nMean probability:",
    round(high_conf_fn["omi_prob"].mean(), 4)
)


# ------------------------------------------------------------
# HIGH-CONFIDENCE FALSE POSITIVES
# ------------------------------------------------------------

print("\n========== HIGH-CONFIDENCE FP ==========")

print(
    "STEMI:",
    int(high_conf_fp["STEMI"].sum())
)

print(
    "NSTEMI:",
    int(high_conf_fp["NSTEMI"].sum())
)

print(
    "LAD:",
    int(high_conf_fp["LAD_GROUP"].sum())
)

print(
    "LCX:",
    int(high_conf_fp["LCX_GROUP"].sum())
)

print(
    "RCA:",
    int(high_conf_fp["RCA_GROUP"].sum())
)

print(
    "\nMean probability:",
    round(high_conf_fp["omi_prob"].mean(), 4)
)

High-confidence false negatives: 24
High-confidence false positives: 134

========== HIGH-CONFIDENCE FN ==========
STEMI: 8
NSTEMI: 16
LAD: 8
LCX: 10
RCA: 11

Mean probability: 0.0796

========== HIGH-CONFIDENCE FP ==========
STEMI: 58
NSTEMI: 14
LAD: 13
LCX: 9
RCA: 13

Mean probability: 0.9015


In [ ]:
groups = {
    "STEMI + OMI":
        df[(df["STEMI"] == 1) & (df["true_OMI"] == 1)],

    "STEMI + non-OMI":
        df[(df["STEMI"] == 1) & (df["true_OMI"] == 0)],

    "NSTEMI + OMI":
        df[(df["NSTEMI"] == 1) & (df["true_OMI"] == 1)],

    "NSTEMI + non-OMI":
        df[(df["NSTEMI"] == 1) & (df["true_OMI"] == 0)]
}


for name, g in groups.items():

    print(f"\n{name}")
    print("-" * 40)

    print("Cases :", len(g))

    print(
        "Mean probability   :",
        round(g["omi_prob"].mean(), 4)
    )

    print(
        "Median probability :",
        round(g["omi_prob"].median(), 4)
    )

    print(
        "25th percentile    :",
        round(g["omi_prob"].quantile(0.25), 4)
    )

    print(
        "75th percentile    :",
        round(g["omi_prob"].quantile(0.75), 4)
    )


STEMI + OMI
----------------------------------------
Cases : 99
Mean probability   : 0.7364
Median probability : 0.8694
25th percentile    : 0.5538
75th percentile    : 0.9654

STEMI + non-OMI
----------------------------------------
Cases : 119
Mean probability   : 0.6436
Median probability : 0.7822
25th percentile    : 0.3242
75th percentile    : 0.93

NSTEMI + OMI
----------------------------------------
Cases : 67
Mean probability   : 0.4899
Median probability : 0.5056
25th percentile    : 0.2291
75th percentile    : 0.7443

NSTEMI + non-OMI
----------------------------------------
Cases : 99
Mean probability   : 0.342
Median probability : 0.2298
25th percentile    : 0.0728
75th percentile    : 0.5758


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def subgroup_performance(name, data):

    y = data["true_OMI"].values
    p = data["omi_prob"].values

    print(f"\n{name}")
    print("-" * 40)

    print("Cases:", len(data))
    print("OMI:", int(y.sum()))
    print("Non-OMI:", int(len(y) - y.sum()))

    print(
        "AUROC:",
        round(roc_auc_score(y, p), 4)
    )

    print(
        "AUPRC:",
        round(average_precision_score(y, p), 4)
    )


stemi_all = df[df["STEMI"] == 1]

nstemi_all = df[df["NSTEMI"] == 1]


subgroup_performance(
    "STEMI patients",
    stemi_all
)

subgroup_performance(
    "NSTEMI patients",
    nstemi_all
)


STEMI patients
----------------------------------------
Cases: 218
OMI: 99
Non-OMI: 119
AUROC: 0.6066
AUPRC: 0.5985

NSTEMI patients
----------------------------------------
Cases: 166
OMI: 67
Non-OMI: 99
AUROC: 0.6329
AUPRC: 0.5389


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

hard_cohort = df[
    (df["STEMI"] == 1) |
    (df["NSTEMI"] == 1)
].copy()

y = hard_cohort["true_OMI"].values
p = hard_cohort["omi_prob"].values

print("Hard cohort cases :", len(hard_cohort))
print("OMI               :", int(y.sum()))
print("Non-OMI           :", int(len(y) - y.sum()))

print(
    "OMI prevalence    :",
    round(y.mean(), 4)
)

print(
    "AUROC             :",
    round(roc_auc_score(y, p), 4)
)

print(
    "AUPRC             :",
    round(average_precision_score(y, p), 4)
)

Hard cohort cases : 384
OMI               : 166
Non-OMI           : 218
OMI prevalence    : 0.4323
AUROC             : 0.617
AUPRC             : 0.5685


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

# All true OMI
positives = df[df["true_OMI"] == 1].copy()

# Hard negatives = STEMI or NSTEMI but NOT OMI
hard_neg = df[
    (df["true_OMI"] == 0) &
    (
        (df["STEMI"] == 1) |
        (df["NSTEMI"] == 1)
    )
].copy()

# Easy negatives = neither STEMI nor NSTEMI and NOT OMI
easy_neg = df[
    (df["true_OMI"] == 0) &
    (df["STEMI"] == 0) &
    (df["NSTEMI"] == 0)
].copy()


def compare_against_negatives(name, negatives):

    tmp = pd.concat([
        positives,
        negatives
    ])

    y = tmp["true_OMI"].values
    p = tmp["omi_prob"].values

    print(f"\n{name}")
    print("-" * 40)

    print("OMI:", len(positives))
    print("Negatives:", len(negatives))

    print(
        "AUROC:",
        round(roc_auc_score(y, p), 4)
    )

    print(
        "AUPRC:",
        round(average_precision_score(y, p), 4)
    )

    print(
        "Negative median probability:",
        round(negatives["omi_prob"].median(), 4)
    )


compare_against_negatives(
    "OMI vs ACS-like negatives",
    hard_neg
)

compare_against_negatives(
    "OMI vs easy negatives",
    easy_neg
)


OMI vs ACS-like negatives
----------------------------------------
OMI: 166
Negatives: 218
AUROC: 0.617
AUPRC: 0.5685
Negative median probability: 0.5032

OMI vs easy negatives
----------------------------------------
OMI: 166
Negatives: 2301
AUROC: 0.9037
AUPRC: 0.545
Negative median probability: 0.0326


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

def bootstrap_auc(data, n_boot=2000, seed=42):

    rng = np.random.default_rng(seed)

    y = data["true_OMI"].values
    p = data["omi_prob"].values

    aucs = []

    for _ in range(n_boot):

        idx = rng.integers(
            0,
            len(data),
            len(data)
        )

        y_b = y[idx]
        p_b = p[idx]

        # skip invalid resample
        if len(np.unique(y_b)) < 2:
            continue

        aucs.append(
            roc_auc_score(y_b, p_b)
        )

    aucs = np.array(aucs)

    return (
        roc_auc_score(y, p),
        np.percentile(aucs, 2.5),
        np.percentile(aucs, 97.5)
    )


# Overall
overall = df.copy()

# Hard cohort
hard = pd.concat([
    positives,
    hard_neg
])

# Easy-negative comparison
easy = pd.concat([
    positives,
    easy_neg
])


for name, data in [
    ("Overall validation", overall),
    ("OMI vs ACS-like negatives", hard),
    ("OMI vs easy negatives", easy)
]:

    auc, lo, hi = bootstrap_auc(data)

    print(f"\n{name}")
    print("-" * 40)

    print("AUROC:", round(auc, 4))
    print(
        "95% CI:",
        f"{lo:.4f} - {hi:.4f}"
    )


Overall validation
----------------------------------------
AUROC: 0.8789
95% CI: 0.8538 - 0.9012

OMI vs ACS-like negatives
----------------------------------------
AUROC: 0.617
95% CI: 0.5595 - 0.6712

OMI vs easy negatives
----------------------------------------
AUROC: 0.9037
95% CI: 0.8807 - 0.9250


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

df["ACS"] = (
    (df["STEMI"] == 1) |
    (df["NSTEMI"] == 1)
).astype(int)

y_acs = df["ACS"].values
omi_score = df["omi_prob"].values

print("Total ECGs :", len(df))
print("ACS cases  :", int(y_acs.sum()))
print("Non-ACS    :", int(len(df) - y_acs.sum()))
print("Prevalence :", round(y_acs.mean(), 4))

print(
    "\nAUROC using OMI score for ACS:",
    round(
        roc_auc_score(y_acs, omi_score),
        4
    )
)

print(
    "AUPRC using OMI score for ACS:",
    round(
        average_precision_score(y_acs, omi_score),
        4
    )
)

acs = df[df["ACS"] == 1]
non_acs = df[df["ACS"] == 0]

print(
    "\nACS median OMI score     :",
    round(acs["omi_prob"].median(), 4)
)

print(
    "Non-ACS median OMI score :",
    round(non_acs["omi_prob"].median(), 4)
)

Total ECGs : 2685
ACS cases  : 384
Non-ACS    : 2301
Prevalence : 0.143

AUROC using OMI score for ACS: 0.8704
AUPRC using OMI score for ACS: 0.6043

ACS median OMI score     : 0.6147
Non-ACS median OMI score : 0.0326


In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/ECG_Project"

keywords = [
    "time", "hour", "hours", "day", "days",
    "ecg", "angio", "angiography", "cath",
    "pci", "admission", "onset"
]

for root, dirs, files in os.walk(ROOT):

    for file in files:

        if not file.lower().endswith(".csv"):
            continue

        path = os.path.join(root, file)

        try:
            temp = pd.read_csv(
                path,
                nrows=3
            )

            matches = [
                c for c in temp.columns
                if any(
                    k in c.lower()
                    for k in keywords
                )
            ]

            if matches:

                print("\nFILE:")
                print(path)

                print("Possible timing columns:")
                print(matches)

        except:
            pass


FILE:
/content/drive/MyDrive/ECG_Project/ACS19955/audit/train_metadata_audit.csv
Possible timing columns:
['ecg_row_record', 'ecg_med_record', 'Time_Interval', 'PCI', 'Prior_PCI']

FILE:
/content/drive/MyDrive/ECG_Project/ACS19955/audit/test_metadata_audit.csv
Possible timing columns:
['ecg_row_record', 'ecg_med_record', 'Time_Interval']

FILE:
/content/drive/MyDrive/ECG_Project/ACS19955/labels/CSV/test.csv
Possible timing columns:
['ecg_row_record', 'ecg_med_record', 'Time_Interval']

FILE:
/content/drive/MyDrive/ECG_Project/ACS19955/labels/CSV/train.csv
Possible timing columns:
['ecg_row_record', 'ecg_med_record', 'Time_Interval', 'PCI', 'Prior_PCI']

FILE:
/content/drive/MyDrive/ECG_Project/ACS19955/median_preparation/metadata/CSV/test.csv
Possible timing columns:
['ecg_row_record', 'ecg_med_record', 'Time_Interval']

FILE:
/content/drive/MyDrive/ECG_Project/ACS19955/median_preparation/metadata/CSV/train.csv
Possible timing columns:
['ecg_row_record', 'ecg_med_record', 'Time_Interv

KeyboardInterrupt: 

In [ ]:
import pandas as pd

META_PATH = (
    "/content/drive/MyDrive/ECG_Project/"
    "ACS19955/audit/train_metadata_audit.csv"
)

meta = pd.read_csv(
    META_PATH,
    dtype=str
)

print("Rows:", len(meta))

print("\nAll columns:")
print(meta.columns.tolist())

print("\nRelevant columns:")
cols = [
    c for c in [
        "record_id",
        "Patient_id",
        "ecg_row_record",
        "ecg_med_record",
        "Time_Interval",
        "PCI",
        "Prior_PCI"
    ]
    if c in meta.columns
]

print(meta[cols].head(10).to_string(index=False))

print("\nTime_Interval examples:")
print(
    meta["Time_Interval"]
    .dropna()
    .head(20)
    .tolist()
)

print("\nMissing Time_Interval:")
print(meta["Time_Interval"].isna().sum())

Rows: 17960

All columns:
['Patient_id', 'ecg_row_record', 'ecg_med_record', 'gender', 'age', 'Time_Interval', 'AMI', 'OMI', 'CTO', 'PCI', 'NSTEMI', 'STEMI', 'UA', 'LM', 'PLAD', 'MLAD', 'DLAD', 'DB', 'PLCX', 'MLCX', 'DLCX', 'OM', 'PRCA', 'MRCA', 'DRCA', 'VF_VT', 'Paced', 'Prior_PCI']

Relevant columns:
Patient_id ecg_row_record ecg_med_record Time_Interval PCI Prior_PCI
    P00001      04904.dat      04904.med            67   0         0
    P00002      15878.dat      15878.med          2895   1         1
    P00002      13502.dat      13502.med          1633   1         0
    P00003      02934.dat      02934.med           440   1         0
    P00004      12776.dat      12776.med           232   0         0
    P00005      15847.dat      15847.med           206   0         0
    P00006      04699.dat      04699.med           502   0         0
    P00007      19037.dat      19037.med          6227   1         0
    P00008      16145.dat      16145.med          5863   0         0
    P0

In [ ]:
print("Validation record IDs:")
print(
    val_meta["record_id"]
    .astype(str)
    .head(10)
    .tolist()
)

print("\nMetadata ECG row IDs:")
print(
    meta["ecg_row_record"]
    .astype(str)
    .head(10)
    .tolist()
)

Validation record IDs:
['12776', '03019', '05748', '03326', '02721', '13862', '15529', '13058', '05058', '14799']

Metadata ECG row IDs:
['04904.dat', '15878.dat', '13502.dat', '02934.dat', '12776.dat', '15847.dat', '04699.dat', '19037.dat', '16145.dat', '03019.dat']


In [ ]:
import pandas as pd

META_PATH = (
    "/content/drive/MyDrive/ECG_Project/"
    "ACS19955/audit/train_metadata_audit.csv"
)

# Load original metadata
meta = pd.read_csv(
    META_PATH,
    dtype=str
)

# Clean metadata record ID
meta["record_id_clean"] = (
    meta["ecg_row_record"]
    .astype(str)
    .str.replace(".dat", "", regex=False)
    .str.zfill(5)
)

# Clean validation record ID
df["record_id"] = (
    df["record_id"]
    .astype(str)
    .str.zfill(5)
)

# Keep timing information
timing_meta = meta[
    [
        "record_id_clean",
        "Time_Interval",
        "PCI",
        "Prior_PCI"
    ]
].copy()

# Merge timing with our prediction/analysis table
df_time = df.merge(
    timing_meta,
    left_on="record_id",
    right_on="record_id_clean",
    how="left"
)

print("Validation ECGs:", len(df_time))
print(
    "Timing available:",
    df_time["Time_Interval"].notna().sum()
)

print(
    "Timing missing:",
    df_time["Time_Interval"].isna().sum()
)

print(
    df_time[
        ["record_id", "true_OMI", "Time_Interval"]
    ].head(10)
)

Validation ECGs: 2685
Timing available: 2685
Timing missing: 0
  record_id  true_OMI Time_Interval
0     12776         0           232
1     03019         0          1623
2     05748         0          4168
3     03326         0          1595
4     02721         0          1824
5     13862         0           872
6     15529         0           428
7     13058         0           193
8     05058         0          3102
9     14799         0           103


In [ ]:
df_time["Time_Interval_num"] = pd.to_numeric(
    df_time["Time_Interval"],
    errors="coerce"
)

print("Available timing:",
      df_time["Time_Interval_num"].notna().sum())

print("Missing timing:",
      df_time["Time_Interval_num"].isna().sum())

print("Negative values:",
      (df_time["Time_Interval_num"] < 0).sum())

print("\nRaw Time_Interval")
print("--------------------")

print("Minimum:",
      df_time["Time_Interval_num"].min())

print("Median:",
      df_time["Time_Interval_num"].median())

print("Maximum:",
      df_time["Time_Interval_num"].max())


# Provisionally interpret as minutes
df_time["time_hours"] = (
    df_time["Time_Interval_num"] / 60
)

print("\nTime in hours")
print("--------------------")

print("Minimum:",
      round(df_time["time_hours"].min(), 2))

print("Median:",
      round(df_time["time_hours"].median(), 2))

print("Maximum:",
      round(df_time["time_hours"].max(), 2))

print("\nPercentiles:")

print(
    df_time["time_hours"]
    .quantile([
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ])
    .round(2)
)

Available timing: 2685
Missing timing: 0
Negative values: 0

Raw Time_Interval
--------------------
Minimum: 5
Median: 1659.0
Maximum: 9989

Time in hours
--------------------
Minimum: 0.08
Median: 27.65
Maximum: 166.48

Percentiles:
0.10      3.19
0.25      7.55
0.50     27.65
0.75     67.42
0.90     99.80
0.95    123.86
0.99    156.02
Name: time_hours, dtype: float64


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# ---------------------------------------------------------
# Create timing groups
# ---------------------------------------------------------

bins = [0, 6, 12, 24, 72, 168]

labels = [
    "<6h",
    "6-12h",
    "12-24h",
    "24-72h",
    "72h-7d"
]

df_time["time_group"] = pd.cut(
    df_time["time_hours"],
    bins=bins,
    labels=labels,
    right=False
)

# ---------------------------------------------------------
# Calculate performance
# ---------------------------------------------------------

results = []

for group in labels:

    g = df_time[
        df_time["time_group"] == group
    ].copy()

    y = g["true_OMI"].values
    p = g["omi_prob"].values

    pred = (p >= 0.5).astype(int)

    positives = y.sum()

    sensitivity = (
        ((pred == 1) & (y == 1)).sum() / positives
        if positives > 0
        else np.nan
    )

    auc = (
        roc_auc_score(y, p)
        if len(np.unique(y)) == 2
        else np.nan
    )

    auprc = (
        average_precision_score(y, p)
        if positives > 0
        else np.nan
    )

    results.append({

        "Time": group,

        "Cases": len(g),

        "OMI": int(positives),

        "Prevalence":
            positives / len(g),

        "AUROC":
            auc,

        "AUPRC":
            auprc,

        "Sensitivity":
            sensitivity
    })


time_results = pd.DataFrame(results)

print(
    time_results.round(4).to_string(index=False)
)

  Time  Cases  OMI  Prevalence  AUROC  AUPRC  Sensitivity
   <6h    550   73      0.1327 0.8523 0.5335       0.6849
 6-12h    248   20      0.0806 0.8268 0.4024       0.6000
12-24h    365   18      0.0493 0.8663 0.4019       0.6667
24-72h    950   36      0.0379 0.9171 0.3259       0.7500
72h-7d    572   19      0.0332 0.8643 0.2027       0.5263


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# Hard ACS-like cohort only
hard_time = df_time[
    (df_time["STEMI"] == 1) |
    (df_time["NSTEMI"] == 1)
].copy()

results = []

for group in labels:

    g = hard_time[
        hard_time["time_group"] == group
    ]

    y = g["true_OMI"].values
    p = g["omi_prob"].values

    omi_n = int(y.sum())
    non_omi_n = int(len(y) - omi_n)

    auc = (
        roc_auc_score(y, p)
        if omi_n > 0 and non_omi_n > 0
        else np.nan
    )

    results.append({
        "Time": group,
        "Cases": len(g),
        "OMI": omi_n,
        "Non_OMI": non_omi_n,
        "AUROC": auc
    })

hard_time_results = pd.DataFrame(results)

print(
    hard_time_results
    .round(4)
    .to_string(index=False)
)

  Time  Cases  OMI  Non_OMI  AUROC
   <6h    153   73       80 0.6135
 6-12h     57   20       37 0.6284
12-24h     42   18       24 0.7106
24-72h     88   36       52 0.6325
72h-7d     44   19       25 0.5011


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

# ---------------------------------------------------------
# Split hard ACS cohort
# ---------------------------------------------------------

early = hard_time[
    hard_time["time_hours"] < 72
].copy()

late = hard_time[
    hard_time["time_hours"] >= 72
].copy()


def bootstrap_auc_ci(data, n_boot=3000, seed=42):

    y = data["true_OMI"].values
    p = data["omi_prob"].values

    rng = np.random.default_rng(seed)

    aucs = []

    for _ in range(n_boot):

        idx = rng.integers(
            0,
            len(data),
            len(data)
        )

        y_b = y[idx]
        p_b = p[idx]

        if len(np.unique(y_b)) < 2:
            continue

        aucs.append(
            roc_auc_score(y_b, p_b)
        )

    aucs = np.array(aucs)

    return (
        roc_auc_score(y, p),
        np.percentile(aucs, 2.5),
        np.percentile(aucs, 97.5)
    )


for name, g in [
    ("<72 hours", early),
    ("72 hours–7 days", late)
]:

    auc, low, high = bootstrap_auc_ci(g)

    print(f"\n{name}")
    print("-" * 40)

    print("Cases   :", len(g))
    print("OMI     :", int(g["true_OMI"].sum()))
    print(
        "Non-OMI :",
        int(len(g) - g["true_OMI"].sum())
    )

    print("AUROC   :", round(auc, 4))

    print(
        "95% CI  :",
        f"{low:.4f} - {high:.4f}"
    )


<72 hours
----------------------------------------
Cases   : 340
OMI     : 147
Non-OMI : 193
AUROC   : 0.631
95% CI  : 0.5690 - 0.6903

72 hours–7 days
----------------------------------------
Cases   : 44
OMI     : 19
Non-OMI : 25
AUROC   : 0.5011
95% CI  : 0.3209 - 0.6759
